In [ ]:
import os
import rawpy
from PIL import Image
import piexif
from tqdm import tqdm
import numpy as np

def extract_gps_from_exif(exif_dict):
    """
    Extract GPS IFD and Image DateTime tag from EXIF dict.
    Returns a minimal EXIF dict with these tags only.
    """
    gps_ifd = exif_dict.get("GPS", {})

    zeroth_ifd = {}
    # Tag 0x0132 is DateTime in 0th IFD
    datetime_tag = 0x0132
    if datetime_tag in exif_dict.get("0th", {}):
        zeroth_ifd[datetime_tag] = exif_dict["0th"][datetime_tag]

    return {
        "0th": zeroth_ifd,
        "GPS": gps_ifd
    }
input_dir = "D:\\Rothaut_Thesis\\Mai\\raw"
output_dir = "D:\\Rothaut_Thesis\\Mai\\jpg"
os.makedirs(output_dir, exist_ok=True)

for filename in tqdm(sorted(os.listdir(input_dir))):
    if not filename.lower().endswith(".dng"):
        continue
    
    filename = filename.replace("._","")
    out_name = filename.replace(".dng", ".jpg").replace(".DNG", ".jpg")
    jpgs = os.listdir(output_dir)
    if out_name in jpgs:
        continue
    dng_path = input_dir +"/"+filename
    with rawpy.imread(dng_path) as raw:
        rgb = raw.postprocess(exp_shift=2, no_auto_bright=True)
    img = Image.fromarray(rgb)
    exif_dict = piexif.load(dng_path)
    gps_only_exif = extract_gps_from_exif(exif_dict)
    exif_bytes = piexif.dump(gps_only_exif)
    out_path = os.path.join(output_dir, out_name)
    img.save(out_path, "jpeg", exif=exif_bytes, quality=100)

print("GPS-only EXIF transfer complete.")

100%|██████████| 495/495 [09:58<00:00,  1.21s/it]

GPS-only EXIF transfer complete.


In [69]:
from PIL import Image
import numpy as np
def convert2to1(img1, img2, indir, outdir):
    numb = img1.split("-")[1]
    image1 = Image.open(os.path.join(indir, img1)).convert("L")
    image2 = Image.open(os.path.join(indir, img2)).convert("L")
    image1_array = np.array(image1)
    image2_array = np.array(image2)
    newg = 150
    image2_array[image2_array != 255] = 0
    image1_array[image1_array != 255] = 0
    image2_array[image2_array == 255] = newg
    merged = np.maximum(image1_array, image2_array)
    merged_im = Image.fromarray(merged, mode="L")
    merged_im.save(os.path.join(outdir,f"mask_{numb}.png"))

indir ="E:\Masterthesis_FIRO\Mai\masks"
outdir="C:/Users/dmz-user/Desktop/mask_gen/thesis/images/masks"
img1list = []
img2list = []
for filename in sorted(os.listdir(indir)):
    if filename.lower().endswith(".png"):
        if "-mit" in filename:
            img2list.append(filename)
        else:
            img1list.append(filename)
for i in range(len(img1list)):
    convert2to1(img1list[i], img2list[i], indir, outdir)